# Open rechunked ECCOFS Icechunk stores from OSN

Native (non-virtual) Icechunk stores of ECCOFS averaged output, rechunked to ~10 MB chunks and stored on the USGS OSN pod.
They are publicly readable, so **no credentials** are needed.

| store | variables | chunks `(time, s_rho, eta, xi)` | good for |
|---|---|---|---|
| `eccofs-avg-temp-balanced-10mb` | `temp` | (17, 5, 144, 167) | mixed access |
| `eccofs-avg-temp-map-10mb` | `temp` | (3, 50, 120, 139) | full-depth maps / profiles |
| `eccofs-avg-uv-balanced-10mb` | `u`, `v` | (17, 5, 144, 167) | mixed access |
| `eccofs-avg-uv-map-10mb` | `u`, `v` | (3, 50, 120, 139) | full-depth maps / profiles |

In [ ]:
import icechunk
import xarray as xr

ENDPOINT = "https://usgs.osn.mghpcc.org"
BUCKET = "esip"

def open_eccofs(name):
    """Open s3://esip/rsignell/<name> anonymously as an xarray Dataset (lazy, dask-backed)."""
    storage = icechunk.s3_storage(
        bucket=BUCKET, prefix=f"rsignell/{name}", region="us-east-1",
        endpoint_url=ENDPOINT, anonymous=True, force_path_style=True,
    )
    repo = icechunk.Repository.open(storage)
    session = repo.readonly_session("main")
    return xr.open_zarr(session.store, consolidated=False)

In [ ]:
stores = ["eccofs-avg-temp-balanced-10mb", "eccofs-avg-temp-map-10mb",
          "eccofs-avg-uv-balanced-10mb", "eccofs-avg-uv-map-10mb"]
dsets = {n: open_eccofs(n) for n in stores}
for n, ds in dsets.items():
    print(n, dict(ds.sizes))
    for v in ds.data_vars:
        print("   ", v, ds[v].dims, "chunks:", ds[v].encoding.get("chunks"))

In [ ]:
ds_temp = dsets["eccofs-avg-temp-map-10mb"]
ds_temp

## Surface temperature map

The `map` chunking keeps all 50 `s_rho` levels in one chunk, so a single-time map reads few chunks. The surface is the last `s_rho` index.

In [ ]:
surf = ds_temp.temp.isel(time=-1, s_rho=-1).load()
surf.plot(figsize=(9, 6), cmap="turbo")

## Time series at a point (balanced chunking)

In [ ]:
temp_bal = dsets["eccofs-avg-temp-balanced-10mb"].temp
ts = temp_bal.isel(s_rho=-1, eta_rho=700, xi_rho=800).load()
ts.plot(figsize=(9, 3))

## Currents

In [ ]:
ds_uv = dsets["eccofs-avg-uv-map-10mb"]
u = ds_uv.u.isel(time=-1, s_rho=-1).load()
u.plot(figsize=(9, 6), cmap="RdBu_r", center=0)